In [1]:
import numpy as np
import json
import pandas as pd

In [2]:
X = np.load("../../datasets/Merged/data/X.npy", allow_pickle=True)

y_depression = np.load("../../datasets/Merged/data/y_depression.npy", allow_pickle=True)

y_stress = np.load("../../datasets/Merged/data/y_stress.npy", allow_pickle=True)

subjects = np.load("../../datasets/Merged/data/subjects.npy", allow_pickle=True)

datasets = np.load("../../datasets/Merged/data/datasets.npy", allow_pickle=True)

with open("../../datasets/Merged/data/metadata.json", "r") as f:
    metadata = json.load(f)

print("Merged dataset loaded successfully!")

Merged dataset loaded successfully!


In [3]:
# Initialize the unified label vector
y = np.full(len(X), -1, dtype=np.int8)

In [4]:
print(y.shape)
print(np.unique(y))

(55157,)
[-1]


In [5]:
# MODMA Healthy -> Healthy
y[y_depression == 0] = 0

# MODMA Depressed -> Depressed
y[y_depression == 1] = 1

# Stress Relax -> Healthy
y[y_stress == 0] = 0

# Stress Active -> Stressed
y[y_stress == 1] = 2

In [6]:
labels, counts = np.unique(y, return_counts=True)

label_map = {
    0: "Healthy",
    1: "Depressed",
    2: "Stressed"
}

summary_df = pd.DataFrame({
    "Class": [label_map[i] for i in labels],
    "Count": counts
})

summary_df

,Class,Count
0,Healthy,21204
1,Depressed,3599
2,Stressed,30354


In [7]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("subjects shape:", subjects.shape)
print("datasets shape:", datasets.shape)

print("\nUnique labels:", np.unique(y))

assert len(X) == len(y) == len(subjects) == len(datasets)

print("\n✅ Dataset validation passed!")

X shape: (55157, 6, 512)
y shape: (55157,)
subjects shape: (55157,)
datasets shape: (55157,)

Unique labels: [0 1 2]

✅ Dataset validation passed!


In [8]:
import os

save_path = "../../datasets/Unified"

os.makedirs(save_path, exist_ok=True)

# Save arrays
np.save(f"{save_path}/X.npy", X)
np.save(f"{save_path}/y.npy", y)
np.save(f"{save_path}/subjects.npy", subjects)
np.save(f"{save_path}/datasets.npy", datasets)

In [9]:
metadata_unified = {
    "summary": {
        "total_segments": int(len(X)),
        "channels": int(X.shape[1]),
        "window_size_samples": int(X.shape[2]),
        "sampling_rate_hz": metadata["summary"]["sampling_rate_hz"]
    },

    "classes": {
        "0": {
            "name": "Healthy",
            "count": int(np.sum(y == 0))
        },
        "1": {
            "name": "Depressed",
            "count": int(np.sum(y == 1))
        },
        "2": {
            "name": "Stressed",
            "count": int(np.sum(y == 2))
        }
    },

    "datasets": metadata["datasets"]
}

In [10]:
with open(f"{save_path}/metadata.json", "w") as f:
    json.dump(metadata_unified, f, indent=4)

print("✅ Unified dataset saved successfully!")

✅ Unified dataset saved successfully!
